# Convolutional Neural Network

### Importing the libraries

In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

In [9]:
tf.__version__

'2.20.0'

## Part 1 - Data Preprocessing

why  transformation? 
- to prevent overfitting 
- high train accuracy and low test accuracy if not 


In [10]:
# Data augmentation for training set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    rotation_range=20,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)

training_set = train_datagen.flow_from_directory(
    '001dataset/dataset/training_set',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

Found 8000 images belonging to 2 classes.


### Preprocessing TestSet

In [11]:
# Only rescaling for test set
test_datagen = ImageDataGenerator(rescale=1./255)

test_set = test_datagen.flow_from_directory(
    '001dataset/dataset/test_set',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.


## Part 2 - Building the CNN

Initializing the CNN

In [12]:
cnn = tf.keras.models.Sequential()

Step 1 - Convolution and pooling

In [13]:
# 1st Convolution + Pooling
cnn.add(tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
cnn.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

/home/kygiet/anaconda3/envs/MLapps/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1762591095.878925   18299 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1762591095.887940   18299 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [14]:
# 2nd Convolution + Pooling
cnn.add(tf.keras.layers.Conv2D(64, (3,3), activation='relu'))
cnn.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

In [15]:
# 3rd Convolution + Pooling
cnn.add(tf.keras.layers.Conv2D(128, (3,3), activation='relu'))
cnn.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

### Step 2 - Flattening 

In [16]:
cnn.add(tf.keras.layers.Flatten())

### step 4 - Full Connection 

In [17]:
# Full connection + Dropout
cnn.add(tf.keras.layers.Dense(128, activation='relu'))
cnn.add(tf.keras.layers.Dropout(0.5))

### step 5 - Output layer

In [18]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [19]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [20]:
# EarlyStopping to prevent overfitting
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

### Training the CNN on the Training Set and evaluationg it on the Test Set 

In [21]:
cnn.fit(
    x=training_set,
    validation_data=test_set,
    epochs=50,  # Train longer
    callbacks=[es]
)

/home/kygiet/anaconda3/envs/MLapps/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50


2025-11-08 14:23:17.883346: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 65028096 exceeds 10% of free system memory.


  1/250 ━━━━━━━━━━━━━━━━━━━━ 7:38 2s/step - accuracy: 0.5000 - loss: 0.7047

2025-11-08 14:23:18.133237: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 32514048 exceeds 10% of free system memory.
2025-11-08 14:23:18.133301: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 65028096 exceeds 10% of free system memory.
2025-11-08 14:23:18.213092: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 65028096 exceeds 10% of free system memory.


  2/250 ━━━━━━━━━━━━━━━━━━━━ 1:08 274ms/step - accuracy: 0.4688 - loss: 0.8193

2025-11-08 14:23:18.409134: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 32514048 exceeds 10% of free system memory.


250/250 ━━━━━━━━━━━━━━━━━━━━ 76s 299ms/step - accuracy: 0.5270 - loss: 0.7012 - val_accuracy: 0.6100 - val_loss: 0.6577
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 73s 293ms/step - accuracy: 0.6395 - loss: 0.6435 - val_accuracy: 0.6980 - val_loss: 0.5764
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 288ms/step - accuracy: 0.6770 - loss: 0.5996 - val_accuracy: 0.7155 - val_loss: 0.5622
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 68s 271ms/step - accuracy: 0.7185 - loss: 0.5557 - val_accuracy: 0.7635 - val_loss: 0.4886
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 68s 270ms/step - accuracy: 0.7368 - loss: 0.5289 - val_accuracy: 0.7765 - val_loss: 0.4773
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 74s 294ms/step - accuracy: 0.7533 - loss: 0.4995 - val_accuracy: 0.7940 - val_loss: 0.4337
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 287ms/step - accuracy: 0.7787 - loss: 0.4732 - val_accuracy: 0.8020 - val_loss: 0.4313
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 287ms/step - accuracy: 0.7893 - loss: 0.4533 - val

## Making a single Prediction

In [22]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    test_image = image.load_img(img_path, target_size=(128,128))
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis=0)/255.0
    result = cnn.predict(test_image)
    prediction = 'dog' if result[0][0] > 0.5 else 'cat'
    print(f"Prediction: {prediction}, Confidence: {result[0][0]:.3f}")

# Example usage
predict_image('./cat.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
Prediction: cat, Confidence: 0.118


In [23]:
print(training_set.class_indices)
print(training_set.samples)


{'cats': 0, 'dogs': 1}
8000


# Confusion matrix

In [24]:
from sklearn import metrics
y_true = test_set.classes
y_pred_prob = cnn.predict(test_set)
y_pred = (y_pred_prob > 0.5).astype(int).reshape(-1)
print("Confusion Matrix:")
print(metrics.confusion_matrix(y_true, y_pred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 59ms/step
Confusion Matrix:
[[478 522]
 [457 543]]


In [30]:
cnn.save("cat_dog_classifier.keras")
